# Exercise 1: Linear Regression with 200 Predictors — Multiple Comparison Problem

Consider the following setup: we generate a dataset with 200 predictors (`X1, ..., X200`) and one response variable `Y`, all drawn independently from a standard normal distribution. We then fit an OLS linear regression of `Y` on all 200 predictors.

## Setup: Replicate the R script in Python

- `pt = 201` total columns (200 predictors + 1 response)
- `p = 200` predictors
- `n = 30 * p = 6000` observations
- All values drawn i.i.d. from N(0, 1), so **Y is independent of all X**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

np.random.seed(0)

pt = 201          # total columns
p  = pt - 1       # number of predictors
n  = 30 * p       # sample size = 6000

data = np.random.randn(n, pt)
cols = [f'X{i}' for i in range(1, p + 1)] + ['Y']
D = pd.DataFrame(data, columns=cols)

print(f'Shape: {D.shape}  (n={n}, p={p})')
D.head()

**Q1.** What does the script above do? What is the true distribution for the random variable $Y$ given the first 200 columns of `D` ($X_1, \ldots, X_{200}$)? Give the name and parameters of this distribution.

## Question 1

**What does the script do?**  
It generates a random dataset of `n = 6000` observations and `p = 200` predictors plus one response `Y`. Every entry is drawn i.i.d. from N(0, 1), so **Y is completely independent of X1, …, X200**.

**True distribution of Y given X1, …, X200:**  
$$Y \mid X_1, \ldots, X_{200} \sim \mathcal{N}(0, 1)$$
The conditional distribution is the same as the marginal — no predictor carries any information about Y.

**Q2.** Write an equation defining the model estimated by the `lm` command. What is the difference between this model and the one defined in Q1?


## Question 2: The estimated model

The `lm` (OLS) command estimates:
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_{200} X_{200} + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

**Difference with Q1:** The estimated model *assumes* that Y depends linearly on the predictors through unknown coefficients $\beta_j$. In reality (Q1) all $\beta_j = 0$ and there is no relationship. The estimated model introduces 201 free parameters that do not exist in the true data-generating process.

In [ ]:
# Fit OLS with statsmodels
formula = 'Y ~ ' + ' + '.join([f'X{i}' for i in range(1, p + 1)])
reg = smf.ols(formula, data=D).fit()
print(reg.summary())

**Q3.** Provide and execute some R code to compute the number of coefficients assessed as significantly non-zero at level 5%. (Hint: `summary(reg)$coefficients`)


## Question 3: Count coefficients significant at 5%

In [ ]:
# Extract p-values for all predictors (exclude intercept)
pvalues = reg.pvalues.drop('Intercept', errors='ignore')

n_significant = (pvalues < 0.05).sum()
print(f'Number of predictors significant at 5%: {n_significant} out of {p}')
print(f'Proportion: {n_significant / p:.1%}')
print()
print('Significant predictors:')
print(pvalues[pvalues < 0.05].sort_values())

**Q4.** Provide an explanation for the result obtained in Q3.


## Question 4: Explanation

Even though **no predictor is truly related to Y**, we perform **200 simultaneous hypothesis tests**, each at the 5% level. By definition, a test at level 5% has a 5% probability of falsely rejecting the null hypothesis when it is true (Type I error). With 200 independent tests, we expect:

$$\text{Expected false positives} = 200 \times 0.05 = 10$$

The result above is fully consistent with this expectation — approximately 10 predictors appear "significant" purely by chance.

**Q5.** What issue is raised by the result obtained in Q3?


## Question 5: The issue raised

This is the **multiple comparisons problem** (also called the multiple testing problem). When many hypotheses are tested simultaneously, the probability of at least one false positive explodes:

$$P(\text{at least one false positive}) = 1 - (1-0.05)^{200} \approx 1 - 4 \times 10^{-5} \approx 100\%$$

Naively trusting the individual p-values leads to spurious conclusions. A model selected on these criteria would overfit severely to noise.

**Q6.** Describe a possible solution to solve this problem, explaining why you think it should work (10 lines expected, ~85 characters per line). Implement that solution and comment the results.

---


## Question 6: Solution — Ridge Regression

**Ridge regression** (L2-penalised OLS) solves the multiple comparison / overfitting problem by adding a penalty on the size of the coefficients:

$$\hat{\beta}^{\text{Ridge}} = \arg\min_{\beta} \left[ \sum_{i=1}^n (y_i - x_i^\top \beta)^2 + \lambda \sum_{j=1}^p \beta_j^2 \right]$$

The regularisation parameter $\lambda \geq 0$ controls the trade-off:
- Large $\lambda$: coefficients are shrunk strongly towards 0 — model is simpler, lower variance, higher bias.
- $\lambda = 0$: recovers ordinary OLS.

**Why it works here:** With 200 predictors and a true model with all $\beta_j = 0$, OLS estimates will be small but noisy. Ridge shrinks all estimates towards zero, recovering something close to the true (null) model. It does not produce individual p-values and therefore does not generate spurious significance.

The optimal $\lambda$ is chosen by **cross-validation** (`RidgeCV`).

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X = D.drop(columns='Y').values
y = D['Y'].values

# Standardise predictors (good practice for penalised regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Choose lambda via cross-validation
alphas = np.logspace(-2, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_scaled, y)
best_alpha = ridge_cv.alpha_
print(f'Best lambda (alpha) chosen by CV: {best_alpha:.4f}')

# Fit Ridge with best alpha
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_scaled, y)

# Compare coefficient distributions
ols_coefs = reg.params.drop('Intercept').values
ridge_coefs = ridge.coef_

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ols_coefs, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title(f'OLS coefficients (std={ols_coefs.std():.4f})')
axes[0].set_xlabel('Coefficient value')

axes[1].hist(ridge_coefs, bins=30, color='darkorange', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title(f'Ridge coefficients (std={ridge_coefs.std():.4f})')
axes[1].set_xlabel('Coefficient value')

plt.suptitle('Ridge shrinks all coefficients towards zero', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nOLS  coef range: [{ols_coefs.min():.4f}, {ols_coefs.max():.4f}]')
print(f'Ridge coef range: [{ridge_coefs.min():.4f}, {ridge_coefs.max():.4f}]')
print('Ridge coefficients are much closer to 0, consistent with the true model.')